# Verify committed analysis artifacts (Tier A)

This notebook does not train models and does not call an LLM.

It checks that files listed in `replication/HASHES.md` still match git.
Tier B (regenerate tables from eval banks) is documented in `replication/README.md`
and is blocked until `SCA2_EVAL_URL` points at a real public zip.

Survey microdata are not in this repository.


In [ ]:
from pathlib import Path
import hashlib
import json

cwd = Path.cwd()
if (cwd / "HASHES.md").exists():
    REPO = cwd.parent
elif (cwd / "replication" / "HASHES.md").exists():
    REPO = cwd
else:
    raise FileNotFoundError("Run from the repository root or replication/")

HASHES = REPO / "replication" / "HASHES.md"


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(65536), b""):
            digest.update(chunk)
    return digest.hexdigest()


expected = {}
for line in HASHES.read_text().splitlines():
    if len(line) > 66 and line[64:66] == "  ":
        expected[line[66:].strip()] = line[:64]

missing, mismatch, ok = [], [], []
for rel, digest in sorted(expected.items()):
    path = REPO / rel
    if not path.exists():
        missing.append(rel)
        continue
    got = sha256(path)
    if got != digest:
        mismatch.append({"path": rel, "expected": digest[:12], "got": got[:12]})
    else:
        ok.append(rel)

print(json.dumps({"ok": len(ok), "missing": missing, "mismatch": mismatch}, indent=2))
if missing or mismatch:
    raise SystemExit("Tier A failed.")
print("Tier A passed.")


## Tier B (not run here)

`analysis/phase2/reproduce_tables.py` rebuilds tables from option-probability CSVs.
Those CSVs are gitignored. There is no default public URL.

If you are on a lab machine with `data/phase2/raw/wvs/` already populated and
`data/wvs_eval_full/*.parquet` rebuilt from WVSA files:

```python
# not executed in this notebook
# !env -u PYTHONPATH python analysis/phase2/reproduce_tables.py
```

Do not point this notebook at Hugging Face adapters. That repo is private and is
not required to check committed tables.
